In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv('dataset_mood_smartphone.csv', index_col = 0)
df.head()

df['time'] = pd.to_datetime(df['time'])

print(df.shape[0])

In [ ]:
# Duration-type variable labels
duration_labels = [
    "screen",
    "appCat.builtin", "appCat.communication", "appCat.entertainment",
    "appCat.finance", "appCat.game", "appCat.office", "appCat.other",
    "appCat.social", "appCat.travel", "appCat.unknown",
    "appCat.utilities", "appCat.weather"
]

# Detect negative durations
neg_durations = df[
    (df["variable"].isin(duration_labels)) &
    (df["value"] < 0)
]

print(f"Negative duration rows found: {neg_durations.shape[0]}")
print(neg_durations[["id", "variable", "value"]].head())

# Remove them
df_no_neg = df.drop(index=neg_durations.index)

print(df_no_neg.shape[0])


In [ ]:
# Identify exact duplicates
duplicates_mask = df_no_neg.duplicated(keep=False)
duplicate_rows = df_no_neg[duplicates_mask].sort_values(by=["id", "time", "variable"])

print(f"Exact duplicate rows: {duplicate_rows.shape[0]}")
print(duplicate_rows.head(10))

# Remove duplicates (keep first)
df_no_neg_dup = df_no_neg.drop_duplicates(keep="first")

print(f"Rows after removing duplicates: {df_no_neg_dup.shape[0]}")


In [ ]:

# Get unique user IDs
user_ids = df_no_neg_dup["id"].unique()
print(f"Found {len(user_ids)} unique users.")

# Collect all real violations
all_violations = []

for user in user_ids:
    print(f"Processing user: {user}")
    
    screen_df = df_no_neg_dup[
        (df_no_neg_dup["variable"] == "screen") &
        (df_no_neg_dup["id"] == user)
    ]
    
    app_df = df_no_neg_dup[
        (df_no_neg_dup["variable"].isin(duration_labels)) &
        (df_no_neg_dup["variable"] != "screen") &
        (df_no_neg_dup["id"] == user)
    ]

    invalid_app_rows = []

    for idx, row in screen_df.iterrows():
        start_time = row["time"]
        duration = row["value"]
        end_time = start_time + pd.to_timedelta(duration, unit="s")

        apps_in_window = app_df[
            (app_df["time"] >= start_time) &
            (app_df["time"] <= end_time)
        ].copy()

        apps_in_window["screen_start"] = start_time
        apps_in_window["screen_end"] = end_time
        apps_in_window["screen_duration"] = duration

        over_limit = apps_in_window[apps_in_window["value"] > duration]

        if not over_limit.empty:
            invalid_app_rows.append(over_limit)

    if invalid_app_rows:
        result_df = pd.concat(invalid_app_rows)
        result_df["overuse"] = result_df["value"] - result_df["screen_duration"]
        real_violations = result_df[result_df["overuse"] > 5]
        all_violations.append(real_violations)

# Combine all violations across users
final_violations = pd.concat(all_violations)

# Remove them from df_cleaned
before = df_no_neg_dup.shape[0]
df_cleaned = df_no_neg_dup.drop(index=final_violations.index)
after = df_cleaned.shape[0]

# Report
print(f"Removed {before - after} rows with impossible app durations")
print(df_cleaned.shape[0])


In [ ]:
print(f"Total violations found (any overuse > 0): {sum([len(df) for df in all_violations])}")
print(df_cleaned.shape[0])


In [ ]:
# Mood
mood_outliers = df_cleaned[
    (df_cleaned["variable"] == "mood") &
    ((df_cleaned["value"] < 1) | (df_cleaned["value"] > 10))
]

# Activity
activity_outliers = df_cleaned[
    (df_cleaned["variable"] == "activity") &
    ((df_cleaned["value"] < 0) | (df_cleaned["value"] > 1))
]

# Arousal / valence
arousal_outliers = df_cleaned[
    (df_cleaned["variable"] == "circumplex.arousal") &
    ((df_cleaned["value"] < -2) | (df_cleaned["value"] > 2))
]

valence_outliers = df_cleaned[
    (df_cleaned["variable"] == "circumplex.valence") &
    ((df_cleaned["value"] < -2) | (df_cleaned["value"] > 2))
]

# Call / SMS
binary_outliers = df_cleaned[
    (df_cleaned["variable"].isin(["call", "sms"])) &
    (~df_cleaned["value"].isin([0, 1]))
]

print(f"Mood outliers: {mood_outliers.shape[0]}")
print(f"Activity outliers: {activity_outliers.shape[0]}")
print(f"Arousal outliers: {arousal_outliers.shape[0]}")
print(f"Valence outliers: {valence_outliers.shape[0]}")
print(f"Call/SMS outliers: {binary_outliers.shape[0]}")